# Day 10 — Wire It All Together

> ⚠️ **Why this matters.** Today everything you built in Week 2 (modules, requests, JSON, caching) becomes ONE TOOL. By the end of today you have a real, online english-helper that auto-fills word entries from a real dictionary, caches results locally, and gracefully degrades when the internet is out.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/quiz/blob/main/phase-1-python-cli/lessons/10-wire-it-together.ipynb)

## What you'll do today

**Time:** 90 min lesson + 90 min build + 30 min quiz.

By the end:

- [ ] `english-helper add WORD` works WITHOUT manually typing IPA or Thai
- [ ] The tool gracefully falls back to manual entry when API is down
- [ ] Cache stats command shows hit rate
- [ ] All Week 2 modules talk to each other via clean function boundaries
- [ ] You can demo a fresh install in under 2 minutes

## The mental model — your app's flow

```mermaid
graph TB
    User[user: english-helper add ubiquitous] --> CLI[cli.py]
    CLI --> CacheCheck[cache.py: get_or_fetch]
    CacheCheck -->|miss| API[api.py: fetch_word]
    CacheCheck -->|hit| Normalize
    API --> Normalize[api.py: normalize]
    Normalize --> Storage[storage.py: save]
    Storage --> Done["'Added.' ✓"]
```

Each box is a module. Each arrow is one function call. **This is what "clean architecture" looks like in 200 lines of Python.**

## 1. The integration layer

You have these modules from Week 2:

- `dictionary.py` (Day 4) — pure data ops on word lists
- `storage.py` (Day 5–6) — load/save the user's vocabulary file
- `api.py` (Day 7–8) — fetch and normalize from the Free Dictionary API
- `cache.py` (Day 9) — disk cache between cli and api
- `cli.py` (Day 5–6) — the REPL / command dispatcher

Today you connect them. **Most of the code today is wiring, not new logic.**

## 2. The updated `add` command flow

In [ ]:
# In cli.py — Week 2's add command
from english_helper.api import fetch_word, to_word_entry
from english_helper.cache import get_or_fetch
from english_helper.storage import load, save
from english_helper.dictionary import add_word as add_to_list  # rename to avoid collision

def cmd_add(args: list[str]) -> str:
    if not args:
        return 'Usage: add WORD'
    word = args[0].lower()
    api_data = get_or_fetch(word, fetch_word)
    if api_data is None:
        return f'No definition for {word!r} (try connecting to the internet)'
    entry = to_word_entry(api_data)
    if not entry.get('ipa'):
        return f'Found {word!r} but no IPA available. Add manually?'
    store = load()
    if word in store:
        return f'{word!r} already in your vocabulary.'
    store[word] = {'ipa': entry['ipa'], 'thai': '', 'definition': entry['definition']}
    save(store)
    return f'Added {word!r}: {entry["ipa"]}'


**Notice:**

- One function = one job (cmd_add only decides what to do, doesn't fetch or store directly)
- Each module exports what's needed; nothing more
- The failure paths are explicit, with friendly messages
- `thai` is empty for auto-added words — user fills in later via `thai WORD <translation>`

## 3. Graceful degradation

In [ ]:
# In api.py — handle 'no network'
import requests

def fetch_word(word: str) -> dict | None:
    url = f'https://api.dictionaryapi.dev/api/v2/entries/en/{word}'
    try:
        r = requests.get(url, timeout=5)
    except (requests.Timeout, requests.ConnectionError) as e:
        # silent return; cli decides what to say
        return None
    if r.status_code == 404:
        return None
    r.raise_for_status()
    data = r.json()
    return data[0] if data else None

**Pattern:** the data layer (`api.py`) returns `None` on any failure. The presentation layer (`cli.py`) decides how to communicate that to the user. Don't `print` from deep inside library code — return data, let the caller choose UX.

> 💡 **Senior engineer principle: "libraries should be silent, callers should communicate."**

## 4. New CLI commands

Three new commands to round out the tool:

**`thai WORD <translation>`** — add Thai translation to an existing word (since API doesn't provide that).

**`cache stats`** — show cache size, hit rate.

**`cache clear`** — empty the cache.

Each is one new function in `cli.py`, ~5 lines.

## End-of-day mini-project — `english-helper add <word>` works end-to-end

> 🎯 **Today's piece:** Phase 1 Week 2 final. Build the integration. Demo your tool.

### Required commands (final list for Week 2)

| Command | Behavior |
|---------|----------|
| `add WORD` | Fetch from API (or cache), add to vocab, leave Thai empty |
| `thai WORD ค่า` | Set Thai translation for an existing word |
| `lookup WORD` | Show full details |
| `list` | List all words |
| `quiz` | Random pronunciation quiz (calls `pronunciation_quiz.py` logic) |
| `cache stats` | Show {entries, size_bytes} |
| `cache clear` | Wipe cache |
| `remove WORD` | Remove from vocab |
| `help`, `quit` | (As before) |

### Acceptance test

Run this exact sequence and confirm output:

```bash
$ rm -rf ~/.english-helper && uv run english-helper
🎯 English Helper — type 'help' for commands
> add ubiquitous
Added 'ubiquitous': /juːˈbɪkwɪtəs/
> thai ubiquitous พบเห็นได้ทั่วไป
Updated thai for ubiquitous.
> lookup ubiquitous
ubiquitous  /juːˈbɪkwɪtəs/  พบเห็นได้ทั่วไป
  found everywhere; existing or being everywhere especially at the same time
> cache stats
Cache: 1 entry, 487 bytes
> add ubiquitous
'ubiquitous' already in your vocabulary.
> quit
Saved.
```

### Stretch

- `audio WORD` — print the audio URL.
- `export FILE` — write the whole vocab as CSV or markdown.
- Add types check via `uv run mypy src/english_helper` — full project passes.
- Friday: have your sibling or a friend install and try it. Watch what confuses them.

## Connect to the project

> 🎯 **End of Phase 1 Week 2.** Your `english-helper` is now a **real online tool**. You can demo it to anyone with: one git clone, one `uv tool install`, one `english-helper add <any-word>`.

> Week 3 (Days 11-15) upgrades this with **classes** — refactor today's loose modules into proper `Word`, `WordStore`, `Quiz` classes, plus a spaced-repetition scheduler that decides what to quiz you on each day.

## Self-check

<details>
<summary>1. Why does <code>fetch_word</code> return None instead of printing the error?</summary>

Separation of concerns. The function knows about HTTP, not UX. The caller (cli.py) knows what to say to the user. Mixing them makes both harder to change and test.
</details>

<details>
<summary>2. What happens to the cache if you delete <code>~/.english-helper</code>?</summary>

Everything's gone — words, cache, all. The directory will be recreated on next save. (For a real product, you'd back this up. For Phase 1, it's fine.)
</details>

<details>
<summary>3. Why is <code>cache stats</code> useful?</summary>

Diagnostic: 'is my cache actually working?'. Also a habit: every system you build should expose health/stats endpoints. Phase 3 (FastAPI) has /health and /metrics for the same reason.
</details>

**Quiz:** [10-wire-it-together-quiz.ipynb](10-wire-it-together-quiz.ipynb)